In [ ]:
from pathlib import Path
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "imagery"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PANORAMA_API = "https://api.data.amsterdam.nl/panorama/panoramas/"

print(PROJECT_ROOT)

In [ ]:

# Preliminary Amsterdam Oost study extent

WEST = 4.900
EAST = 4.930
SOUTH = 52.350
NORTH = 52.368

study_center = [
    (SOUTH + NORTH) / 2,
    (WEST + EAST) / 2,
]

m = folium.Map(
    location=study_center,
    zoom_start=14,
    tiles="OpenStreetMap",
)

folium.Rectangle(
    bounds=[
        [SOUTH, WEST],
        [NORTH, EAST],
    ],
    color="red",
    weight=2,
    fill=False,
    tooltip="Preliminary study area",
).add_to(m)

m

In [ ]:
import numpy as np
import math

GRID_SPACING_M = 150

center_lat = (SOUTH + NORTH) / 2

# Approximate conversion from metres to decimal degrees
lat_step = GRID_SPACING_M / 111_320
lon_step = GRID_SPACING_M / (
    111_320 * math.cos(math.radians(center_lat))
)

lats = np.arange(SOUTH, NORTH + lat_step, lat_step)
lons = np.arange(WEST, EAST + lon_step, lon_step)

grid_records = []

grid_id = 1

for lat in lats:
    for lon in lons:
        grid_records.append({
            "grid_id": f"G{grid_id:03d}",
            "longitude": lon,
            "latitude": lat
        })
        grid_id += 1

grid_df = pd.DataFrame(grid_records)

print("Grid points:", len(grid_df))
grid_df.head()

In [ ]:
m_grid = folium.Map(
    location=study_center,
    zoom_start=14,
    tiles="OpenStreetMap"
)

folium.Rectangle(
    bounds=[[SOUTH, WEST], [NORTH, EAST]],
    color="red",
    weight=2,
    fill=False
).add_to(m_grid)

for _, row in grid_df.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=2,
        weight=1,
        fill=True,
        fill_opacity=0.7
    ).add_to(m_grid)

m_grid

In [ ]:
from datetime import datetime

SEARCH_RADIUS_M = 70

def query_panoramas(lon, lat, radius=SEARCH_RADIUS_M, page_size=100):
    
    params = {
        "near": f"{lon},{lat}",
        "srid": 4326,
        "radius": radius,
        "page_size": page_size,
    }

    r = requests.get(
        PANORAMA_API,
        params=params,
        timeout=30
    )

    r.raise_for_status()

    data = r.json()

    return data.get("_embedded", {}).get("panoramas", [])

In [ ]:
def panorama_priority(pano):
    
    ts = pd.to_datetime(
        pano["timestamp"],
        utc=True
    )

    year = ts.year
    month = ts.month

    growing_season = 4 <= month <= 9

    # Lower number = better
    if year == 2025 and growing_season:
        group = 1
    elif year == 2024 and growing_season:
        group = 2
    elif year == 2023 and growing_season:
        group = 3
    else:
        group = 4

    return (
        group,
        -ts.timestamp()
    )

In [ ]:
def select_best_panorama(panoramas):
    
    if not panoramas:
        return None

    ranked = sorted(
        panoramas,
        key=panorama_priority
    )

    return ranked[0]

In [ ]:
test_results = []

for _, row in grid_df.head(5).iterrows():

    panos = query_panoramas(
        row["longitude"],
        row["latitude"]
    )

    best = select_best_panorama(panos)

    if best is None:
        print(row["grid_id"], "→ no panorama")
        continue

    ts = pd.to_datetime(best["timestamp"], utc=True)

    print(
        row["grid_id"],
        "→",
        ts.date(),
        "|",
        best["pano_id"]
    )

    test_results.append(best)

In [ ]:
def query_panoramas(lon, lat, radius=SEARCH_RADIUS_M, page_size=100):
    """
    Retrieve ALL panoramas within radius, following Amsterdam API pagination.
    """

    params = {
        "near": f"{lon},{lat}",
        "srid": 4326,
        "radius": radius,
        "page_size": page_size,
    }

    panoramas = []
    url = PANORAMA_API

    while url:

        r = requests.get(
            url,
            params=params if url == PANORAMA_API else None,
            timeout=30
        )

        r.raise_for_status()
        data = r.json()

        panoramas.extend(
            data.get("_embedded", {}).get("panoramas", [])
        )

        # Follow API-provided next page
        next_link = (
            data.get("_links", {})
                .get("next", {})
                .get("href")
        )

        url = next_link

    return panoramas

In [ ]:
from time import sleep

candidate_records = []

for i, row in grid_df.iterrows():

    try:
        panos = query_panoramas(
            row["longitude"],
            row["latitude"]
        )

        best = select_best_panorama(panos)

        if best is None:
            continue

        coords = best["geometry"]["coordinates"]
        ts = pd.to_datetime(best["timestamp"], utc=True)

        candidate_records.append({
            "grid_id": row["grid_id"],

            "grid_lon": row["longitude"],
            "grid_lat": row["latitude"],

            "pano_id": best["pano_id"],
            "longitude": coords[0],
            "latitude": coords[1],

            "timestamp": ts,
            "year": ts.year,
            "month": ts.month,

            "heading": best.get("heading"),

            "image_url": (
                best["_links"]
                ["equirectangular_medium"]
                ["href"]
            )
        })

        print(
            f"{i+1:03d}/{len(grid_df)}",
            row["grid_id"],
            "→",
            ts.date()
        )

        # Be polite to the API
        sleep(0.05)

    except Exception as e:
        print(
            f"{row['grid_id']} FAILED:",
            e
        )

candidate_df = pd.DataFrame(candidate_records)

print("\nCompleted")
print("Grid points:", len(grid_df))
print("Candidates found:", len(candidate_df))

In [ ]:
print("Candidate panoramas:", len(candidate_df))

print("\nYear distribution:")
print(
    candidate_df["year"]
    .value_counts()
    .sort_index()
)

print("\nMonth distribution:")
print(
    candidate_df["month"]
    .value_counts()
    .sort_index()
)

print(
    "\nUnique panorama IDs:",
    candidate_df["pano_id"].nunique()
)

In [ ]:
candidate_unique = (
    candidate_df
    .sort_values(
        ["year", "timestamp"],
        ascending=False
    )
    .drop_duplicates(
        subset="pano_id",
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    "Before deduplication:",
    len(candidate_df)
)

print(
    "After deduplication:",
    len(candidate_unique)
)

In [ ]:
m_candidates = folium.Map(
    location=study_center,
    zoom_start=14,
    tiles="OpenStreetMap"
)

folium.Rectangle(
    bounds=[[SOUTH, WEST], [NORTH, EAST]],
    color="red",
    weight=2,
    fill=False
).add_to(m_candidates)

for _, row in candidate_unique.iterrows():

    popup_text = (
        f"<b>{row['grid_id']}</b><br>"
        f"Date: {row['timestamp'].date()}<br>"
        f"Year: {row['year']}<br>"
        f"Pano: {row['pano_id']}"
    )

    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=4,
        weight=1,
        fill=True,
        fill_opacity=0.8,
        popup=popup_text
    ).add_to(m_candidates)

m_candidates

In [ ]:
from shapely.geometry import Point

candidate_gdf = gpd.GeoDataFrame(
    candidate_unique.copy(),
    geometry=gpd.points_from_xy(
        candidate_unique["longitude"],
        candidate_unique["latitude"]
    ),
    crs="EPSG:4326"
)

candidate_rd = candidate_gdf.to_crs("EPSG:28992")

print(candidate_rd.crs)
print("Candidates:", len(candidate_rd))

In [ ]:
preferred = candidate_rd[
    (candidate_rd["year"] == 2025) &
    (candidate_rd["month"].between(6, 9))
].copy()

print("Preferred candidates:", len(preferred))

In [ ]:
import numpy as np

def farthest_point_sample(gdf, n=50):
    """
    Select n spatially distributed locations using
    greedy farthest-point sampling in a projected CRS.
    """

    coords = np.column_stack([
        gdf.geometry.x.values,
        gdf.geometry.y.values
    ])

    # Start near the geographic centre
    centre = coords.mean(axis=0)

    first_idx = np.argmin(
        np.linalg.norm(coords - centre, axis=1)
    )

    selected = [first_idx]

    min_dist = np.linalg.norm(
        coords - coords[first_idx],
        axis=1
    )

    while len(selected) < min(n, len(gdf)):

        next_idx = np.argmax(min_dist)
        selected.append(next_idx)

        new_dist = np.linalg.norm(
            coords - coords[next_idx],
            axis=1
        )

        min_dist = np.minimum(
            min_dist,
            new_dist
        )

    return gdf.iloc[selected].copy()


sample_rd = farthest_point_sample(
    preferred,
    n=50
)

print("Final sample:", len(sample_rd))

In [ ]:
from scipy.spatial.distance import pdist

coords = np.column_stack([
    sample_rd.geometry.x,
    sample_rd.geometry.y
])

distances = pdist(coords)

print(f"Minimum spacing: {distances.min():.1f} m")
print(f"Median spacing:  {np.median(distances):.1f} m")

In [ ]:
sample_wgs = sample_rd.to_crs("EPSG:4326")

m_sample = folium.Map(
    location=study_center,
    zoom_start=14,
    tiles="OpenStreetMap"
)

folium.Rectangle(
    bounds=[[SOUTH, WEST], [NORTH, EAST]],
    color="red",
    weight=2,
    fill=False
).add_to(m_sample)

for i, (_, row) in enumerate(sample_wgs.iterrows(), start=1):

    folium.CircleMarker(
        location=[
            row.geometry.y,
            row.geometry.x
        ],
        radius=5,
        weight=1,
        fill=True,
        fill_opacity=0.9,
        popup=(
            f"<b>AMS{i:03d}</b><br>"
            f"{row['timestamp'].date()}<br>"
            f"{row['pano_id']}"
        )
    ).add_to(m_sample)

m_sample

In [ ]:
sample_wgs = sample_wgs.reset_index(drop=True)

sample_wgs["location_id"] = [
    f"AMS{i:03d}"
    for i in range(1, len(sample_wgs) + 1)
]

# Put ID first
cols = [
    "location_id",
    "grid_id",
    "pano_id",
    "longitude",
    "latitude",
    "timestamp",
    "year",
    "month",
    "heading",
    "image_url",
    "geometry"
]

sample_wgs = sample_wgs[cols]

csv_path = PROCESSED_DIR / "panorama_locations.csv"
geojson_path = PROCESSED_DIR / "panorama_locations.geojson"

sample_wgs.drop(columns="geometry").to_csv(
    csv_path,
    index=False
)

sample_wgs.to_file(
    geojson_path,
    driver="GeoJSON"
)

print("Saved:")
print(csv_path)
print(geojson_path)

In [ ]:
qa_sample = sample_wgs.sample(
    n=10,
    random_state=42
).copy()

qa_sample[
    ["location_id", "timestamp", "latitude", "longitude"]
]

In [ ]:
from PIL import Image
from io import BytesIO

QA_DIR = PROJECT_ROOT / "data" / "raw" / "imagery" / "qa"
QA_DIR.mkdir(parents=True, exist_ok=True)

def download_panorama(row, output_dir):

    output_path = output_dir / f"{row['location_id']}.jpg"

    if output_path.exists():
        return output_path

    r = requests.get(
        row["image_url"],
        timeout=60
    )

    r.raise_for_status()

    img = Image.open(BytesIO(r.content))
    img.save(output_path, quality=95)

    return output_path


qa_paths = {}

for _, row in qa_sample.iterrows():

    path = download_panorama(
        row,
        QA_DIR
    )

    qa_paths[row["location_id"]] = path

    print(
        row["location_id"],
        "→",
        path.name
    )

In [ ]:
import py360convert
import numpy as np

qa_views = {}

for location_id, path in qa_paths.items():

    pano = Image.open(path).convert("RGB")
    pano_np = np.array(pano)

    view = py360convert.e2p(
        pano_np,
        fov_deg=100,
        u_deg=0,
        v_deg=0,
        out_hw=(512, 512)
    )

    qa_views[location_id] = view

In [ ]:
fig, axes = plt.subplots(
    len(qa_sample),
    2,
    figsize=(10, 4 * len(qa_sample))
)

for i, (_, row) in enumerate(qa_sample.iterrows()):

    location_id = row["location_id"]

    pano = Image.open(
        qa_paths[location_id]
    ).convert("RGB")

    pano_np = np.array(pano)

    for j, angle in enumerate([0, 180]):

        view = py360convert.e2p(
            pano_np,
            fov_deg=100,
            u_deg=angle,
            v_deg=0,
            out_hw=(512, 512)
        )

        axes[i, j].imshow(view)
        axes[i, j].axis("off")
        axes[i, j].set_title(
            f"{location_id} — view {angle}°"
        )

plt.tight_layout()
plt.show()

In [ ]:
PANORAMA_DIR = (
    PROJECT_ROOT /
    "data" /
    "raw" /
    "imagery" /
    "panoramas"
)

PANORAMA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

download_log = []

for _, row in sample_wgs.iterrows():

    location_id = row["location_id"]
    output_path = PANORAMA_DIR / f"{location_id}.jpg"

    try:

        if not output_path.exists():

            r = requests.get(
                row["image_url"],
                timeout=60
            )

            r.raise_for_status()

            img = Image.open(
                BytesIO(r.content)
            ).convert("RGB")

            img.save(
                output_path,
                quality=95
            )

            status = "downloaded"

        else:
            status = "already exists"

        download_log.append({
            "location_id": location_id,
            "status": status,
            "path": str(output_path)
        })

        print(
            location_id,
            "→",
            status
        )

    except Exception as e:

        download_log.append({
            "location_id": location_id,
            "status": "failed",
            "error": str(e)
        })

        print(
            location_id,
            "FAILED:",
            e
        )

In [ ]:
download_log_df = pd.DataFrame(download_log)

print(
    download_log_df["status"]
    .value_counts()
)

print(
    "\nJPEG files:",
    len(list(PANORAMA_DIR.glob("*.jpg")))
)

In [ ]:
# Reload the frozen 50-location sample if needed

SAMPLE_GEOJSON = (
    PROCESSED_DIR /
    "panorama_locations.geojson"
)

if SAMPLE_GEOJSON.exists():

    sample_wgs = gpd.read_file(
        SAMPLE_GEOJSON
    )

    sample_rd = sample_wgs.to_crs(
        "EPSG:28992"
    )

    print(
        f"Loaded existing sample: "
        f"{len(sample_wgs)} locations"
    )

else:

    print(
        "No saved sample found. "
        "Run the sampling section first."
    )

In [ ]:
network_path = (
    PROJECT_ROOT /
    "data" /
    "raw" /
    "gis" /
    "loopfietsnetwerk_edges.geojson"
)

if network_path.exists():
    network_path.unlink()
    print("Deleted incomplete GeoJSON.")

In [ ]:
from io import BytesIO

WFS_URL = (
    "https://api.data.amsterdam.nl/"
    "v1/wfs/loopfietsnetwerk/"
)

# Slightly larger than our study area
BUFFER_DEG = 0.003
WEST  = 4.900
EAST  = 4.930
SOUTH = 52.350
NORTH = 52.368

import requests

EDGES_API = (
    "https://api.data.amsterdam.nl/"
    "v1/loopfietsnetwerk/edges/"
)

test_row = sample_wgs.iloc[0]

lon = test_row.geometry.x
lat = test_row.geometry.y

params = {
    "geometrie[within]": f"POINT({lon} {lat}),50",
    "_format": "geojson",
    "page_size": 100
}

r = requests.get(
    EDGES_API,
    params=params,
    timeout=30
)

print("URL:")
print(r.url)

print("\nHTTP:", r.status_code)
print("Size:", len(r.content), "bytes")

r.raise_for_status()

data = r.json()

print("\nType:", data.get("type"))
print("Features:", len(data.get("features", [])))

In [ ]:
print(r.text[:200])

In [ ]:
features = data.get("features", [])

for feature in features[:10]:

    props = feature["properties"]

    print(
        "ID:", props.get("id"),
        "| class:", props.get("wegklasse"),
        "| street:", props.get("straatnaam")
    )

In [ ]:
from time import sleep

SEARCH_NETWORK_RADIUS_M = 30

all_edge_features = []

for i, (_, row) in enumerate(sample_wgs.iterrows(), start=1):

    location_id = row["location_id"]
    lon = row.geometry.x
    lat = row.geometry.y

    params = {
        "geometrie[within]": f"POINT({lon} {lat}),{SEARCH_NETWORK_RADIUS_M}",
        "_format": "geojson",
        "page_size": 100
    }

    try:

        r = requests.get(
            EDGES_API,
            params=params,
            timeout=30
        )

        r.raise_for_status()

        data = r.json()
        features = data.get("features", [])

        for feature in features:

            feature["properties"]["location_id"] = location_id
            all_edge_features.append(feature)

        print(
            f"{i:02d}/50",
            location_id,
            "→",
            len(features),
            "edges"
        )

        sleep(0.05)

    except Exception as e:

        print(
            location_id,
            "FAILED:",
            e
        )

In [ ]:
from shapely.geometry import shape

records = []

for feature in all_edge_features:

    props = feature["properties"].copy()

    props["geometry"] = shape(
        feature["geometry"]
    )

    records.append(props)

nearby_edges = gpd.GeoDataFrame(
    records,
    geometry="geometry",
    crs="EPSG:4326"
)

print("Total retrieved edge records:", len(nearby_edges))
print("Locations represented:", nearby_edges["location_id"].nunique())

In [ ]:
NEARBY_EDGES_PATH = (
    PROJECT_ROOT /
    "data" /
    "raw" /
    "gis" /
    "panorama_nearby_edges.geojson"
)

nearby_edges.to_file(
    NEARBY_EDGES_PATH,
    driver="GeoJSON"
)

print("Saved:", NEARBY_EDGES_PATH)

In [ ]:
edge_classes = (
    nearby_edges["wegklasse"]
    .value_counts()
    .rename_axis("wegklasse")
    .reset_index(name="count")
)

display(edge_classes)

In [ ]:
print(
    "Total unique edge classes:",
    nearby_edges["wegklasse"].nunique()
)

print(
    "\nEdges with street names:",
    nearby_edges["straatnaam"].notna().sum()
)

print(
    "Edges without street names:",
    nearby_edges["straatnaam"].isna().sum()
)

In [ ]:
ROAD_CLASSES = {
    "osm_primary",
    "osm_primary_link",
    "osm_secondary",
    "osm_tertiary",
    "osm_residential",
    "osm_unclassified",
    "osm_service",
    "osm_busway"
}

road_edges = nearby_edges[
    nearby_edges["wegklasse"].isin(ROAD_CLASSES)
].copy()

print("Road-edge records:", len(road_edges))

print("\nRoad classes:")
print(
    road_edges["wegklasse"]
    .value_counts()
)

print(
    "\nLocations with at least one road:",
    road_edges["location_id"].nunique(),
    "/ 50"
)

In [ ]:
road_rd = road_edges.to_crs("EPSG:28992")

points_rd = sample_wgs.to_crs("EPSG:28992").copy()

point_lookup = (
    points_rd
    .set_index("location_id")
    .geometry
    .to_dict()
)

road_rd["distance_m"] = road_rd.apply(
    lambda row:
        row.geometry.distance(
            point_lookup[row["location_id"]]
        ),
    axis=1
)

In [ ]:
nearest_roads = (
    road_rd
    .sort_values("distance_m")
    .groupby("location_id")
    .first()
    .reset_index()
)

display(
    nearest_roads[
        [
            "location_id",
            "straatnaam",
            "wegklasse",
            "distance_m"
        ]
    ]
    .sort_values("location_id")
)

In [ ]:
print(
    nearest_roads["distance_m"]
    .describe()
)

print("\nSelected road classes:")

print(
    nearest_roads["wegklasse"]
    .value_counts()
)

print("\nLargest matching distances:")

display(
    nearest_roads[
        [
            "location_id",
            "straatnaam",
            "wegklasse",
            "distance_m"
        ]
    ]
    .sort_values(
        "distance_m",
        ascending=False
    )
    .head(10)
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 12)
)

# all nearby road candidates
road_rd.plot(
    ax=ax,
    linewidth=0.5,
    alpha=0.20
)

# selected road edges
nearest_roads.plot(
    ax=ax,
    linewidth=2.0
)

# panorama locations
points_rd.plot(
    ax=ax,
    markersize=25
)

for _, row in points_rd.iterrows():

    ax.annotate(
        row["location_id"],
        (
            row.geometry.x,
            row.geometry.y
        ),
        fontsize=6
    )

plt.title(
    "Panorama locations and selected street edges"
)

plt.axis("equal")
plt.show()

In [ ]:
matched_ids = set(nearest_roads["location_id"])
all_ids = set(sample_wgs["location_id"])

missing_ids = sorted(all_ids - matched_ids)

print("Locations without road match:")
print(missing_ids)

print("\nMatches farther than 10 m:")

display(
    nearest_roads.loc[
        nearest_roads["distance_m"] > 10,
        [
            "location_id",
            "straatnaam",
            "wegklasse",
            "distance_m"
        ]
    ].sort_values("distance_m", ascending=False)
)

In [ ]:
far_ids = nearest_roads.loc[
    nearest_roads["distance_m"] > 10,
    "location_id"
].tolist()

problem_ids = sorted(
    set(missing_ids + far_ids)
)

print("Problem locations:", problem_ids)

for location_id in problem_ids:

    print("\n" + "=" * 60)
    print(location_id)

    display(
        nearby_edges.loc[
            nearby_edges["location_id"] == location_id,
            [
                "wegklasse",
                "straatnaam"
            ]
        ]
        .value_counts()
        .reset_index(name="count")
    )

In [ ]:
# Project all nearby edges
nearby_edges_rd = nearby_edges.to_crs("EPSG:28992")

# Calculate distance from each edge to its corresponding panorama
nearby_edges_rd["distance_m"] = nearby_edges_rd.apply(
    lambda row:
        row.geometry.distance(
            point_lookup[row["location_id"]]
        ),
    axis=1
)

for location_id in problem_ids:

    print("\n" + "=" * 70)
    print(location_id)

    display(
        nearby_edges_rd.loc[
            nearby_edges_rd["location_id"] == location_id,
            [
                "wegklasse",
                "straatnaam",
                "distance_m"
            ]
        ]
        .sort_values("distance_m")
        .head(15)
    )

In [ ]:
final_matches = nearest_roads[
    nearest_roads["distance_m"] <= 10
].copy()

print("Automatically accepted roads:", len(final_matches))

In [ ]:
fallback_rules = {
    "AMS010": "osm_cycleway",
    "AMS023": "osm_pedestrian",
    "AMS024": "osm_pedestrian"
}

fallback_rows = []

for location_id, edge_class in fallback_rules.items():

    candidates = nearby_edges_rd[
        (nearby_edges_rd["location_id"] == location_id) &
        (nearby_edges_rd["wegklasse"] == edge_class)
    ].copy()

    best = candidates.sort_values(
        "distance_m"
    ).iloc[0]

    fallback_rows.append(best)

fallback_gdf = gpd.GeoDataFrame(
    fallback_rows,
    geometry="geometry",
    crs="EPSG:28992"
)

final_matches = pd.concat(
    [
        final_matches,
        fallback_gdf
    ],
    ignore_index=True
)

final_matches = gpd.GeoDataFrame(
    final_matches,
    geometry="geometry",
    crs="EPSG:28992"
)

print("Final matched locations:", len(final_matches))

display(
    final_matches[
        [
            "location_id",
            "straatnaam",
            "wegklasse",
            "distance_m"
        ]
    ].sort_values("location_id")
)

In [ ]:
import math
from shapely.geometry import LineString, MultiLineString


def longest_line(geometry):
    """Return a LineString from LineString/MultiLineString geometry."""

    if isinstance(geometry, LineString):
        return geometry

    if isinstance(geometry, MultiLineString):
        return max(
            geometry.geoms,
            key=lambda g: g.length
        )

    return None


def local_corridor_orientation(
    line_geometry,
    point_geometry,
    offset_m=5
):
    """
    Estimate local corridor orientation near the panorama.

    Returns angle in degrees:
        0   = east
        90  = north
        180 = west
        270 = south

    Note: this is a GIS geographic orientation,
    NOT yet a py360convert u_deg.
    """

    line = longest_line(line_geometry)

    if line is None:
        return None

    position = line.project(point_geometry)

    p1 = line.interpolate(
        max(0, position - offset_m)
    )

    p2 = line.interpolate(
        min(line.length, position + offset_m)
    )

    dx = p2.x - p1.x
    dy = p2.y - p1.y

    angle = math.degrees(
        math.atan2(dy, dx)
    )

    return angle % 360

In [ ]:
final_matches["corridor_angle"] = final_matches.apply(
    lambda row:
        local_corridor_orientation(
            row.geometry,
            point_lookup[row["location_id"]],
            offset_m=5
        ),
    axis=1
)

display(
    final_matches[
        [
            "location_id",
            "straatnaam",
            "wegklasse",
            "distance_m",
            "corridor_angle"
        ]
    ].sort_values("location_id")
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 12)
)

points_rd.plot(
    ax=ax,
    markersize=20
)

arrow_length = 40

for _, row in final_matches.iterrows():

    point = point_lookup[
        row["location_id"]
    ]

    angle_rad = math.radians(
        row["corridor_angle"]
    )

    dx = arrow_length * math.cos(angle_rad)
    dy = arrow_length * math.sin(angle_rad)

    ax.arrow(
        point.x,
        point.y,
        dx,
        dy,
        head_width=8,
        head_length=10,
        length_includes_head=True
    )

    ax.annotate(
        row["location_id"],
        (point.x, point.y),
        fontsize=6
    )

plt.title(
    "Derived local transport-corridor orientations"
)

plt.axis("equal")
plt.show()

In [ ]:
ORIENTATION_PATH = (
    PROCESSED_DIR /
    "panorama_corridor_matches.geojson"
)

final_matches.to_crs("EPSG:4326").to_file(
    ORIENTATION_PATH,
    driver="GeoJSON"
)

print("Saved:", ORIENTATION_PATH)

In [ ]:
test_ids = [
    "AMS014",
    "AMS018",
    "AMS031",
    "AMS040",
    "AMS046",
    "AMS049"
]

In [ ]:
import py360convert
from PIL import Image
import numpy as np

PANORAMA_DIR = (
    PROJECT_ROOT /
    "data" /
    "raw" /
    "imagery" /
    "panoramas"
)
TEST_ANGLES = list(range(0, 360, 45))

fig, axes = plt.subplots(
    len(test_ids),
    len(TEST_ANGLES),
    figsize=(24, 3 * len(test_ids))
)

for row_i, location_id in enumerate(test_ids):

    pano_path = (
        PANORAMA_DIR /
        f"{location_id}.jpg"
    )

    pano = Image.open(
        pano_path
    ).convert("RGB")

    pano_np = np.array(pano)

    for col_i, angle in enumerate(TEST_ANGLES):

        view = py360convert.e2p(
            pano_np,
            fov_deg=80,
            u_deg=angle,
            v_deg=0,
            out_hw=(400, 400)
        )

        # Remove bottom camera/car area
        view = view[:350, :, :]

        ax = axes[row_i, col_i]

        ax.imshow(view)
        ax.axis("off")

        ax.set_title(
            f"{location_id}\n{angle}°",
            fontsize=9
        )

plt.tight_layout()
plt.show()

In [ ]:
display(
    final_matches[
        final_matches["location_id"].isin(test_ids)
    ][
        [
            "location_id",
            "straatnaam",
            "wegklasse",
            "corridor_angle"
        ]
    ]
    .sort_values("location_id")
)

In [ ]:
heading_lookup = (
    sample_wgs[
        ["location_id", "heading"]
    ]
    .drop_duplicates("location_id")
)

calibration_df = final_matches.merge(
    heading_lookup,
    on="location_id",
    how="left"
)

display(
    calibration_df[
        calibration_df["location_id"].isin(test_ids)
    ][
        [
            "location_id",
            "straatnaam",
            "wegklasse",
            "corridor_angle",
            "heading"
        ]
    ]
    .sort_values("location_id")
)

In [ ]:
PREVIEW_DIR = (
    PROCESSED_DIR /
    "orientation_previews"
)

PREVIEW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

angles = [0, 90, 180, 270]

for _, row in sample_wgs.iterrows():

    location_id = row["location_id"]

    pano_path = (
        PANORAMA_DIR /
        f"{location_id}.jpg"
    )

    pano = Image.open(
        pano_path
    ).convert("RGB")

    pano_np = np.array(pano)

    fig, axes = plt.subplots(
        1, 4,
        figsize=(12, 3)
    )

    for ax, angle in zip(axes, angles):

        view = py360convert.e2p(
            pano_np,
            fov_deg=90,
            u_deg=angle,
            v_deg=0,
            out_hw=(400, 400)
        )

        # Crop camera/car from bottom
        view = view[:350]

        ax.imshow(view)
        ax.axis("off")
        ax.set_title(f"{angle}°")

    fig.suptitle(
        location_id,
        fontsize=14
    )

    plt.tight_layout()

    output_path = (
        PREVIEW_DIR /
        f"{location_id}_orientation.jpg"
    )

    plt.savefig(
        output_path,
        dpi=120,
        bbox_inches="tight"
    )

    plt.close()

print(
    "Orientation previews:",
    len(list(PREVIEW_DIR.glob("*.jpg")))
)